# Pipeline Tách Tóc (Hair Segmentation)

Notebook này tái hiện **đúng từng bước** quy trình tách tóc được dùng trong dự án Hair Try-on, cụ thể là khi Admin upload một kiểu tóc mới qua API `/admin/upload-hairstyle`.

**File nguồn trong dự án:**
- `backend/app/services/hair_segmenter.py` — Class `HairSegmenter`
- `backend/app/services/hair_straightener.py` — Hàm `straighten_hair`
- `backend/app/services/hair_extractor.py` — Hàm `extract_hair` → upload Supabase Storage
- `backend/app/api/admin.py` — Endpoint `POST /admin/upload-hairstyle` → insert Supabase DB
- `backend/app/database/db.py` — Supabase client

---
## Tổng quan Pipeline (bao gồm Supabase)

```
Admin gọi POST /admin/upload-hairstyle
             ↓
    [ML Pipeline — hair_extractor.py]
             ↓
[Bước 1] MediaPipe Hair Segmenter → Confidence Map (0.0 – 1.0)
             ↓
[Bước 2] Core Mask  (ngưỡng 0.50 + morphological close/open)
             ↓
[Bước 3] Strands Mask (ngưỡng 0.20 + connected components)
             ↓
[Bước 4] Binary Mask = Core | Strands
             ↓
[Bước 5] Soft Alpha Channel (gamma = 0.65)
             ↓
[Bước 6] RGBA PNG bytes (trong bộ nhớ)
             ↓
[Bước 7] Nắn thẳng tóc (Polynomial / PCA)
             ↓
[Bước 8] Upload lên Supabase Storage (bucket: hairstyles)
             ↓  CDN public URL
[Bước 9] INSERT vào Supabase DB (table: hairstyles)
             ↓
    Trả HairstyleResponse về Admin
```

## Cài đặt thư viện

In [ ]:
# !pip install mediapipe opencv-python-headless pillow numpy matplotlib supabase

## Import thư viện

In [ ]:
import io
import os
import re
import uuid
import urllib.request
import warnings
warnings.filterwarnings('ignore')

import cv2
import mediapipe as mp
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

print(f"✓ MediaPipe: {mp.__version__}")
print(f"✓ OpenCV:    {cv2.__version__}")

---
## Bước 0 — Tải ảnh đầu vào

Sử dụng ảnh tóc có sẵn trong dự án. Thay `IMAGE_PATH` bằng ảnh bất kỳ.

In [ ]:
IMAGE_PATH = "backend/assets/hairstyles/t1.png"

input_pil = Image.open(IMAGE_PATH).convert("RGB")
input_arr = np.array(input_pil)

print(f"Kích thước ảnh: {input_pil.width} x {input_pil.height} px")

plt.figure(figsize=(5, 5))
plt.imshow(input_arr)
plt.title("Ảnh tóc đầu vào", fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()

---
## Bước 1 — Tải mô hình MediaPipe Hair Segmenter

**Công nghệ:** Google MediaPipe Image Segmenter (TensorFlow Lite)

- Model `hair_segmenter.tflite` (float32, ~3MB) — được Google train riêng cho task phân đoạn tóc
- Output: **confidence mask** — ma trận `H×W` với giá trị `0.0` (nền) → `1.0` (tóc)
- Chạy hoàn toàn trên **CPU**, không cần GPU
- `output_confidence_masks=True` → trả probability thay vì category label

**File:** `hair_segmenter.py` lines 15–43

In [ ]:
MODELS_DIR = os.environ.get("MODELS_DIR", ".")
MODEL_PATH = os.path.join(MODELS_DIR, "hair_segmenter.tflite")
MODEL_URL  = (
    "https://storage.googleapis.com/mediapipe-models/"
    "image_segmenter/hair_segmenter/float32/latest/hair_segmenter.tflite"
)

os.makedirs(MODELS_DIR, exist_ok=True)
if not os.path.exists(MODEL_PATH):
    print("Đang tải model...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("✓ Xong")
else:
    print(f"✓ Model đã có: {MODEL_PATH}")

options = vision.ImageSegmenterOptions(
    base_options=python.BaseOptions(model_asset_path=MODEL_PATH),
    output_confidence_masks=True,
)
segmenter = vision.ImageSegmenter.create_from_options(options)
print("✓ HairSegmenter sẵn sàng")

---
## Bước 2 — Chạy model → lấy Confidence Map

**Công nghệ:** `ImageSegmenter.segment()`

- Input: `mp.Image` format `SRGB`
- `confidence_masks[1]` = lớp tóc (index 0 = background)
- `np.squeeze` chuyển tensor `(1, H, W)` → `(H, W)`

**File:** `hair_segmenter.py` → `_run_model()` lines 45–57

In [ ]:
buf = io.BytesIO()
input_pil.save(buf, format="PNG")
image_bytes = buf.getvalue()

img_rgb = np.array(Image.open(io.BytesIO(image_bytes)).convert("RGB"))
mp_img  = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
result  = segmenter.segment(mp_img)

conf = np.squeeze(
    np.array(result.confidence_masks[1].numpy_view(), dtype=np.float32)
)

print(f"Confidence map: {conf.shape}, min={conf.min():.3f}, max={conf.max():.3f}")
print(f"Vùng tóc (conf>0.5): {(conf>0.5).sum():,} px ({(conf>0.5).mean()*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img_rgb)
axes[0].set_title("Ảnh gốc (RGB)", fontsize=13)
axes[0].axis('off')
im = axes[1].imshow(conf, cmap='hot', vmin=0, vmax=1)
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
axes[1].set_title("Confidence Map (0=nền, 1=tóc)", fontsize=13)
axes[1].axis('off')
plt.suptitle("Bước 2: Output MediaPipe Hair Segmenter", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Bước 3 — Tạo Core Mask

**Công nghệ:** Binary Threshold + Morphological Operations (OpenCV)

| Phép toán | Kernel | Mục đích |
|---|---|---|
| `MORPH_CLOSE` ×2 | Ellipse 9×9 | Lấp lỗ hổng bên trong vùng tóc |
| `MORPH_OPEN` ×1 | Ellipse 9×9 | Loại đốm nhiễu nhỏ bên ngoài |

**File:** `hair_segmenter.py` → `_make_core()` lines 59–68

In [ ]:
T_CORE = 0.50
raw_core = (conf > T_CORE).astype(np.uint8) * 255
k9 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
after_close = cv2.morphologyEx(raw_core, cv2.MORPH_CLOSE, k9, iterations=2)
core = cv2.morphologyEx(after_close, cv2.MORPH_OPEN, k9)

print(f"Raw (conf>0.5):  {(raw_core>0).sum():,} px")
print(f"Sau CLOSE:       {(after_close>0).sum():,} px")
print(f"Core mask:       {(core>0).sum():,} px")

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, img, title in zip(axes,
    [conf, raw_core, after_close, core],
    ["Confidence", f"Binary>{T_CORE}", "Sau CLOSE×2", "Core Mask"]):
    ax.imshow(img, cmap='gray' if img.max()>1 else 'hot', vmin=0, vmax=255 if img.max()>1 else 1)
    ax.set_title(title, fontsize=11); ax.axis('off')
plt.suptitle("Bước 3: Core Mask", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Bước 4 — Tạo Strands Mask

**Công nghệ:** Connected Components Analysis

- Ngưỡng 0.2 bắt sợi tóc mỏng nhưng sinh nhiều nhiễu
- **Giải pháp:** chỉ giữ blob có kết nối vật lý với core → loại nhiễu nền

**File:** `hair_segmenter.py` → `_make_strands()` lines 71–94

In [ ]:
T_STRAND = 0.20
k3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
raw_strand = (conf > T_STRAND).astype(np.uint8) * 255
raw_strand = cv2.morphologyEx(raw_strand, cv2.MORPH_CLOSE, k3)

combined = cv2.bitwise_or(raw_strand, core)
_, labels = cv2.connectedComponents(combined)
n_labels  = labels.max()

core_labels = set(int(v) for v in np.unique(labels[core > 0]) if v != 0)
keep = np.zeros_like(raw_strand)
for lbl in core_labels:
    keep[labels == lbl] = 255

strands = cv2.bitwise_and(keep, raw_strand)
strands = cv2.bitwise_and(strands, cv2.bitwise_not(core))

print(f"Tổng blobs: {n_labels} | Core labels: {core_labels}")
print(f"Strands thêm: {(strands>0).sum():,} px")

labels_vis = np.zeros((*labels.shape, 3), dtype=np.uint8)
np.random.seed(42)
for i in range(1, n_labels + 1):
    labels_vis[labels == i] = np.random.randint(50, 255, 3)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, img, title in zip(axes,
    [raw_strand, labels_vis, keep, strands],
    [f"Raw conf>{T_STRAND}", f"CC ({n_labels} blobs)", "Blobs→Core", "Strands Mask"]):
    ax.imshow(img, cmap='gray' if img.ndim==2 else None)
    ax.set_title(title, fontsize=11); ax.axis('off')
plt.suptitle("Bước 4: Strands Mask", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Bước 5 — Binary Mask = Core | Strands

**File:** `hair_segmenter.py` → `get_hair_mask()` line 116

In [ ]:
binary_mask = cv2.bitwise_or(core, strands)
print(f"Core: {(core>0).sum():,} px | Strands: {(strands>0).sum():,} px | Total: {(binary_mask>0).sum():,} px")

compare = np.zeros((*binary_mask.shape, 3), dtype=np.uint8)
compare[core>0]    = [255, 100, 0]
compare[strands>0] = [0, 150, 255]
overlay = img_rgb.copy()
overlay[binary_mask>0] = (overlay[binary_mask>0].astype(np.float32)*0.5 + np.array([0,200,100])*0.5).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_rgb); axes[0].set_title("Ảnh gốc"); axes[0].axis('off')
axes[1].imshow(compare)
axes[1].legend(handles=[
    mpatches.Patch(color=(1,.4,0), label='Core (>0.5)'),
    mpatches.Patch(color=(0,.6,1), label='Strands (>0.2)'),
], loc='lower right', fontsize=9)
axes[1].set_title("Core vs Strands"); axes[1].axis('off')
axes[2].imshow(overlay); axes[2].set_title("Overlay"); axes[2].axis('off')
plt.suptitle("Bước 5: Binary Mask = Core | Strands", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Bước 6 — Soft Alpha Channel (gamma = 0.65)

**Công thức:**
```
alpha = conf^0.65 × 255   [trong vùng binary > 0]
alpha = max(alpha, 255)    [conf > 0.8 → đục hoàn toàn]
```
- `γ < 1` nâng giá trị thấp lên → sợi tóc rìa mờ dần tự nhiên thay vì cắt cứng

**File:** `hair_segmenter.py` → `_soft_alpha()` lines 96–106

In [ ]:
GAMMA = 0.65
soft  = np.clip(conf.astype(np.float32) ** GAMMA, 0, 1) * 255
alpha = np.where(binary_mask > 0, soft, 0).astype(np.uint8)
alpha = np.maximum(alpha, (conf > 0.8).astype(np.uint8) * 255)

x, y = np.linspace(0,1,200), np.linspace(0,1,200)**GAMMA
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(binary_mask, cmap='gray'); axes[0].set_title("Binary (cứng)"); axes[0].axis('off')
axes[1].imshow(alpha, cmap='gray', vmin=0, vmax=255); axes[1].set_title(f"Soft Alpha (γ={GAMMA})"); axes[1].axis('off')
axes[2].plot(x, x, 'b--', lw=1.5, label='γ=1'); axes[2].plot(x, y, 'r-', lw=2.5, label=f'γ={GAMMA}')
for vx, lbl, c in [(0.5,'T_CORE','gray'),(0.2,'T_STRAND','orange'),(0.8,'Floor','green')]:
    axes[2].axvline(vx, color=c, linestyle=':', alpha=0.7, label=lbl)
axes[2].set(xlabel='Confidence', ylabel='Alpha', title='Gamma Curve'); axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)
plt.suptitle("Bước 6: Soft Alpha Channel", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()
print(f"Alpha: min={alpha.min()}, max={alpha.max()}, mean={alpha.mean():.1f}")

---
## Bước 7 — RGBA Image + Nắn thẳng tóc

**Tại sao cần nắn thẳng?**  
Ảnh tóc upload ở nhiều góc khác nhau. Chuẩn hóa về chiều dọc giúp bước ghép tóc sau này tính scale/rotation nhất quán.

**Thuật toán (`hair_straightener.py`):**
1. Lấy bottom 25% của tóc → `polyfit(xs, ys, 1)` → slope
2. `angle = arctan(slope)` → xoay
3. Fallback PCA nếu `|angle| > 45°`, bỏ qua nếu `|angle| < 1.5°`

In [ ]:
# ── Tạo RGBA ──────────────────────────────────────────────────────────────────
rgba = np.dstack([img_rgb, alpha]).astype(np.uint8)
hair_rgba_pil = Image.fromarray(rgba, "RGBA")
print(f"RGBA: {rgba.shape}, tóc px: {(rgba[:,:,3]>0).sum():,}")

def on_bg(rgba_arr, bg):
    a = rgba_arr[:,:,3:4].astype(np.float32)/255
    return (rgba_arr[:,:,:3]*a + np.array(bg)*(1-a)).clip(0,255).astype(np.uint8)

# ── Nắn thẳng ────────────────────────────────────────────────────────────────
img_arr  = rgba
alpha_ch = img_arr[:,:,3]
ys, xs   = np.where(alpha_ch > 30)
angle, method = 0.0, "skip"
straight_pil = hair_rgba_pil

if len(ys) > 0:
    ymin, ymax = ys.min(), ys.max()
    xmin, xmax = xs.min(), xs.max()
    b_thresh = ymin + int((ymax - ymin) * 0.75)
    b_ys, b_xs = np.where((alpha_ch > 30) & (np.arange(alpha_ch.shape[0])[:,None] >= b_thresh))
    if len(b_xs) >= 10:
        slope, _ = np.polyfit(b_xs, b_ys, 1)
        angle = float(np.degrees(np.arctan(slope)))
        method = "polynomial"
        if abs(angle) > 45:
            pts = np.column_stack((xs,ys)).astype(np.float32)
            _, evec, _ = cv2.PCACompute2(pts, mean=None)
            angle  = float(np.degrees(np.arctan2(evec[0,1], evec[0,0]))) - 90
            method = "PCA"
        if abs(angle) > 35:   angle, method = 0.0, "guard>35°"
        if abs(angle) < 1.5:  angle, method = 0.0, "skip<1.5°"
        if angle != 0.0:
            center = (int((xmin+xmax)/2), int((ymin+ymax)/2))
            straight_pil = hair_rgba_pil.rotate(-angle, resample=Image.BICUBIC, center=center)

straight_arr = np.array(straight_pil)
print(f"Góc: {angle:.2f}° | Phương pháp: {method}")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(on_bg(rgba, [255,255,255]))
axes[0].set_title("Trước nắn thẳng"); axes[0].axis('off')
axes[1].imshow(on_bg(straight_arr, [255,255,255]))
axes[1].set_title(f"Sau nắn thẳng ({angle:.1f}°)"); axes[1].axis('off')
plt.suptitle("Bước 7: RGBA + Nắn thẳng tóc", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Bước 8 — Upload lên Supabase Storage

**Công nghệ:** Supabase Storage (S3-compatible object storage với CDN)

**Luồng trong dự án (`hair_extractor.py` lines 40–55):**

```python
supabase.storage.from_("hairstyles").upload(
    path     = filename,          # e.g. "toc_ngan_a3b2c1d4.png"
    file     = png_bytes,
    file_options = {"contentType": "image/png", "upsert": "true"}
)
public_url = supabase.storage.from_("hairstyles").get_public_url(filename)
# → "https://xxx.supabase.co/storage/v1/object/public/hairstyles/toc_ngan_a3b2c1d4.png"
```

**Tại sao dùng Supabase Storage thay vì lưu local?**
- File ảnh RGBA được phục vụ qua **CDN toàn cầu** — load nhanh hơn từ frontend
- Không chiếm dung lượng server (Render free tier chỉ 1GB disk)
- Dễ scale: Supabase Storage hỗ trợ hàng GB không cần config thêm
- Public URL trực tiếp — frontend không cần proxy qua backend API

**Cấu trúc bucket:**
```
Bucket: hairstyles  (public = true)
  ├── toc_ngan_a3b2c1d4.png
  ├── toc_dai_xoan_e5f6a7b8.png
  └── bob_cut_c9d0e1f2.png
```

In [ ]:
# ── Mô phỏng bước upload (không cần Supabase thật) ───────────────────────────
# Đây là code chạy thật trong hair_extractor.py:

def _safe_filename(name: str) -> str:
    name = name.strip().lower()
    name = re.sub(r"[^\w\s-]", "", name)
    name = re.sub(r"[\s]+", "_", name)
    return name or "hairstyle"

HAIR_NAME = "Tóc Ngắn Layer"
base      = _safe_filename(HAIR_NAME)
uid       = uuid.uuid4().hex[:8]           # 8 ký tự hex random
filename  = f"{base}_{uid}.png"            # unique filename

# Convert thành PNG bytes (đây là gì sẽ upload lên Supabase)
buf = io.BytesIO()
straight_pil.save(buf, format="PNG")
png_bytes = buf.getvalue()

# Mô phỏng public URL mà Supabase trả về
SUPABASE_URL_EXAMPLE = "https://abcxyzproject.supabase.co"
STORAGE_BUCKET = "hairstyles"
simulated_public_url = f"{SUPABASE_URL_EXAMPLE}/storage/v1/object/public/{STORAGE_BUCKET}/{filename}"

print("═" * 55)
print("  SUPABASE STORAGE UPLOAD")
print("═" * 55)
print(f"  Tên kiểu tóc:  '{HAIR_NAME}'")
print(f"  Safe filename: '{base}'")
print(f"  UUID suffix:   '{uid}'")
print(f"  Filename:      '{filename}'")
print(f"  File size:     {len(png_bytes)/1024:.1f} KB")
print(f"  Bucket:        '{STORAGE_BUCKET}' (public)")
print(f"  Public URL:    '{simulated_public_url}'")

print("\n--- Code thực tế trong hair_extractor.py ---")
print("""
supabase = get_supabase()          # singleton Supabase client

supabase.storage.from_(STORAGE_BUCKET).upload(
    path         = filename,       # 'toc_ngan_layer_a3b2c1d4.png'
    file         = png_bytes,      # RGBA PNG bytes từ bộ nhớ
    file_options = {
        "contentType": "image/png",
        "upsert": "true",          # ghi đè nếu trùng tên
    },
)

public_url = supabase.storage.from_(STORAGE_BUCKET).get_public_url(filename)
# → image_path để lưu vào DB
""")

---
## Bước 9 — INSERT vào Supabase Database

**Công nghệ:** Supabase PostgreSQL + Python SDK (`supabase-py 2.x`)

**Schema bảng `hairstyles` (tạo một lần trong Supabase SQL Editor):**
```sql
CREATE TABLE hairstyles (
  id           BIGSERIAL PRIMARY KEY,
  name         TEXT        NOT NULL,
  image_path   TEXT        NOT NULL,   -- Supabase Storage public URL
  preview_path TEXT,
  created_at   TIMESTAMPTZ DEFAULT NOW()
);
```

**Luồng trong dự án (`admin.py` lines 29–40):**
```python
public_url = extract_hair(image_bytes, name)   # từ Bước 8

result = (
    supabase.table("hairstyles")
    .insert({
        "name"       : name,
        "image_path" : public_url,    # CDN URL từ Storage
        "created_at" : datetime.now(utc).isoformat()
    })
    .execute()
)
return result.data[0]   # → HairstyleResponse về Admin
```

**Tại sao lưu public URL vào `image_path`?**
- Frontend (`client.ts`) gọi `hairImageUrl(image_path)` → trả thẳng URL Supabase
- Không cần backend proxy ảnh → giảm tải server
- Khi xóa: parse filename từ URL → `supabase.storage.from_(...).remove([filename])`

In [ ]:
from datetime import datetime, timezone

# ── Mô phỏng INSERT vào Supabase DB ─────────────────────────────────────────
simulated_db_record = {
    "id"           : 7,
    "name"         : HAIR_NAME,
    "image_path"   : simulated_public_url,
    "preview_path" : None,
    "created_at"   : datetime.now(timezone.utc).isoformat(),
}

print("═" * 55)
print("  SUPABASE DB — Table: hairstyles")
print("═" * 55)
for k, v in simulated_db_record.items():
    print(f"  {k:<15}: {v}")

print("\n--- Code thực tế trong admin.py ---")
print("""
result = (
    supabase.table("hairstyles")
    .insert({
        "name"       : name,
        "image_path" : public_url,
        "created_at" : datetime.now(timezone.utc).isoformat()
    })
    .execute()
)
return HairstyleResponse(**result.data[0])
""")

print("\n--- Frontend nhận được ---")
print(f"  image_path = '{simulated_public_url}'")
print("  hairImageUrl(image_path) → trả thẳng URL (Supabase CDN)")
print("  <Image source={{uri: hairImageUrl(item.image_path)}} />")

---
## Tổng kết — Toàn bộ Admin Upload Flow

```
Admin upload ảnh tóc
         ↓
POST /admin/upload-hairstyle (multipart: name + image)
         ↓
   hair_extractor.py
         ├─ HairSegmenter → Confidence Map
         ├─ Core Mask (T=0.5 + morph)
         ├─ Strands Mask (T=0.2 + CC)
         ├─ Soft Alpha (γ=0.65)
         ├─ RGBA PNG bytes
         ├─ Hair Straightening (poly/PCA)
         └─ Supabase Storage.upload() → public_url
         ↓
   admin.py
         └─ supabase.table("hairstyles").insert({name, image_path=public_url})
         ↓
HairstyleResponse {id, name, image_path (CDN URL), created_at}
         ↓
Frontend: hiển thị ảnh trực tiếp từ Supabase CDN
```

In [ ]:
# ── Tổng kết pipeline hoàn chỉnh ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
panels = [
    (img_rgb,                          "Bước 0\nẢnh gốc"),
    (conf,                             "Bước 2\nConfidence Map"),
    (raw_core,                         "Bước 3\nBinary>0.5"),
    (core,                             "Bước 3\nCore Mask"),
    (strands,                          "Bước 4\nStrands Mask"),
    (binary_mask,                      "Bước 5\nBinary Mask"),
    (alpha,                            "Bước 6\nSoft Alpha"),
    (on_bg(straight_arr, [240,240,240]),"Bước 7→8→9\nRGBA → Supabase"),
]
cmaps = [None,'hot','gray','gray','gray','gray','gray',None]
for ax, (img, title), cmap in zip(axes.flat, panels, cmaps):
    ax.imshow(img, cmap=cmap, vmin=0, vmax=255 if (img.max()>1 and cmap=='gray') else (1 if cmap=='hot' else None))
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')
plt.suptitle("Pipeline Tách Tóc → Upload Supabase Storage → INSERT DB",
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()